# AAV2 -- analyse de la technique gagnante : feature auxiliaire de profondeur

Dans `AAV2_viab_profile_model_denoising_v2.ipynb`, sur les 7 techniques testées (baseline, filtre
profondeur, Huber, winsorisation, feature auxiliaire, régularisation, ensembling), **la feature
auxiliaire de profondeur (`log1p(w)` concaténé à l'entrée one-hot, version corrigée après le bug de
fuite de cible) reste la meilleure**, même après correction. Ce notebook creuse spécifiquement ce
modèle-là, au-delà du simple `r` held-out :

1. **Scatter prédit vs réel**, aux vs baseline (même split), côte à côte.
2. **Recovery top-k%**, aux vs baseline.
3. **Comparaison au plafond de bruit** (`AAV2_viab_noise_ceiling.ipynb` : plafond analytique
   +0.744, plafond empirique split-half +0.819).
4. **Le modèle utilise-t-il vraiment `log1p(w)` de façon sensée ?** Deux diagnostics :
   - erreur absolue par quantile de confiance `w` (aux vs baseline) -- si l'idée marche, l'écart
     aux vs baseline devrait être concentré sur les variants les MOINS confiants ;
   - **dépendance partielle** : séquences fixées, on fait varier `log1p(w)` artificiellement en
     entrée et on regarde comment la prédiction bouge -- teste directement l'hypothèse de départ
     (rétrécissement vers la moyenne quand la confiance est basse, écartement vers la vraie valeur
     quand elle est haute).

Même CSV/recette que `AAV2_viab_profile_model_denoising_v2.ipynb` (`AAV2_organoides_sorted.csv`,
`eps=0.5`, `ShallowProfileMLP`, MSE non pondérée). **Pas exécuté automatiquement.**


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Modelization_V2")
sys.path.insert(0, str(_root / "lib"))
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
import jax.numpy as jnp
from flax import nnx
from typing import Optional
import optax
from tqdm.auto import tqdm

from analysisV1 import AA_LABELS, pearson, precision_at_k, plot_topk_recovery

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

L, A = 7, 20

CSV = Path("AAV2_organoides.csv")
if not CSV.exists():
    CSV = _root / "notebooks/notebooks/AAVs dataset/AAV2/AAV2_organoides.csv"
assert CSV.exists(), CSV

VIAB_COL = "log2_enrichissement_virus_sur_plasmide"
use_cols = ["sequence", "compte_plasmide", "compte_virus", VIAB_COL]
dtypes = {"sequence": "string", "compte_plasmide": "float32", "compte_virus": "float32", VIAB_COL: "float32"}
df = pd.read_csv(CSV, usecols=use_cols, dtype=dtypes)
df["sequence"] = df["sequence"].astype("string")
print(f"{len(df):,} lignes chargées depuis {CSV.name}")

plasmid = df["compte_plasmide"].to_numpy(np.float64)
virus   = df["compte_virus"].to_numpy(np.float64)
y_all   = df[VIAB_COL].to_numpy(np.float64)

EPS = 0.5
w_all = 1.0 / (1.0 / (virus + EPS) + 1.0 / (plasmid + EPS))
logw_all = np.log1p(w_all)

lut = np.zeros(256, dtype=np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i
S = lut[np.frombuffer("".join(df["sequence"]).encode("ascii"), np.uint8)].reshape(len(df), L)
oh = lambda idx: np.eye(A, dtype=np.float32)[S[idx]].reshape(len(idx), -1)


### 1. Architecture + boucle d'entraînement (verbatim, MSE non pondérée)

In [ ]:
class ShallowProfileMLP(nnx.Module):
    '''Archi standard du projet (Linear+BatchNorm+Dropout+GELU x2).'''

    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (128, 64),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x: jax.Array, *, train: bool, rngs: Optional[nnx.Rngs] = None) -> jax.Array:
        x = self.linear1(x)
        x = self.batchnorm1(x, use_running_average=not train)
        x = self.dropout1(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)
        x = self.linear2(x)
        x = self.batchnorm2(x, use_running_average=not train)
        x = self.dropout2(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)
        return self.linear3(x).squeeze(-1)


@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)
    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    y_pred = model(x, train=False)
    return jnp.mean((y_pred - y) ** 2)


@nnx.jit
def predict_step(model, x):
    return model(x, train=False)


@nnx.scan(in_axes=(nnx.Carry, 0, 0), out_axes=(nnx.Carry, 0))
def train_epoch_scan(carry, xb, yb):
    model, optimizer, rngs = carry
    loss = train_step(model, optimizer, xb, yb, rngs)
    return (model, optimizer, rngs), loss


def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


def train_mlp(model_cls, model_kwargs, X_train, y_train, X_val, y_val,
              epochs=150, batch_size=512, peak_lr=1e-3, final_lr=1e-5,
              weight_decay=0, patience=5, seed=0, verbose=True):
    rngs  = nnx.Rngs(seed)
    model = model_cls(rngs=rngs, **model_kwargs)

    n_train         = X_train.shape[0]
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps     = steps_per_epoch * epochs

    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr, warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9), end_value=final_lr)
    optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param)

    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val,   y_val   = jnp.asarray(X_val),   jnp.asarray(y_val)

    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs = float("inf"), None, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in tqdm(range(epochs), desc="  epochs", disable=not verbose, leave=False):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm      = jax.random.permutation(perm_key, n_train)
        batch_idx = perm[: steps_per_epoch * batch_size].reshape(steps_per_epoch, batch_size)

        (model, optimizer, rngs), step_losses = train_epoch_scan(
            (model, optimizer, rngs), X_train[batch_idx], y_train[batch_idx])
        train_loss = float(jnp.mean(step_losses))
        val_loss   = float(eval_step(model, X_val, y_val))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val, bad_epochs = val_loss, 0
            best_state = nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    nnx.update(model, best_state)
    return model, history


def batched_predict(model, X_oh, batch_size=16384):
    chunks = []
    for start in range(0, X_oh.shape[0], batch_size):
        chunks.append(np.asarray(predict_step(model, jnp.asarray(X_oh[start:start + batch_size]))))
    return np.concatenate(chunks)


### 2. Entraînement -- baseline (one-hot seul) vs aux (one-hot + `log1p(w)`)

Même split fit/eval et même split train/val (`seed=0`) pour les deux modèles -- seule la présence de la feature de confiance change.

In [ ]:
N_CAP = 300_000
RNG = np.random.default_rng(0)

idx_finite = np.flatnonzero(np.isfinite(y_all))
if len(idx_finite) > N_CAP:
    idx_finite = RNG.choice(idx_finite, N_CAP, replace=False)
fit_i, eval_i = train_test_split(idx_finite, test_size=0.5, random_state=0)

def build_X(idx, aux):
    Xoh = oh(idx)
    if not aux:
        return Xoh
    return np.concatenate([Xoh, logw_all[idx].astype(np.float32).reshape(-1, 1)], axis=1)

y_fit, y_eval = y_all[fit_i], y_all[eval_i]

models, results = {}, {}
for tag, aux in [("baseline", False), ("aux (log1p w)", True)]:
    X_fit  = build_X(fit_i, aux)
    X_eval = build_X(eval_i, aux)
    Xtr, ytr, Xva, yva = split_train_val(X_fit, y_fit, val_frac=0.15, seed=0)
    model, hist = train_mlp(ShallowProfileMLP, dict(input_dim=X_fit.shape[1]), Xtr, ytr, Xva, yva, seed=0, verbose=False)
    pred_eval = batched_predict(model, X_eval)
    r = pearson(y_eval, pred_eval)
    models[tag] = model
    results[tag] = dict(y_eval=y_eval, pred_eval=pred_eval, r=r, n_fit=len(Xtr))
    print(f"[{tag:16s}] n_fit={len(Xtr):,}  epochs={len(hist['train_loss'])}  -> held-out r={r:+.3f}")


### 3. Scatter prédit vs réel (hexbin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, tag in zip(axes, results):
    res = results[tag]
    y, p = res["y_eval"], res["pred_eval"]
    ax.hexbin(y, p, gridsize=55, bins="log", mincnt=1, cmap="viridis")
    lo, hi = min(y.min(), p.min()), max(y.max(), p.max())
    ax.plot([lo, hi], [lo, hi], "w--", lw=0.9)
    ax.set_title(f"{tag}\nr = {res['r']:+.3f}", fontsize=10)
    ax.set_xlabel("log enrichment réel"); ax.set_ylabel("log enrichment prédit")
fig.suptitle("AAV2 viab -- baseline vs feature auxiliaire de profondeur (held-out)", y=1.02)
fig.tight_layout()
plt.show()


### 4. Recovery top-k%

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.8))
for ax, tag in zip(axes, results):
    res = results[tag]
    plot_topk_recovery(res["y_eval"], res["pred_eval"], k_frac=0.10,
                        xlabel="log enrichment réel", ylabel="log enrichment prédit",
                        title=tag, ax=ax)
    ax.legend(fontsize=7)
fig.suptitle("AAV2 viab -- top-10% recovery (held-out)", y=1.02)
fig.tight_layout()
plt.show()

rows = []
for tag, res in results.items():
    row = {"variante": tag, "Pearson r": round(res["r"], 3)}
    for frac in (0.01, 0.05, 0.10, 0.20):
        row[f"top-{int(frac*100)}%"] = round(precision_at_k(res["y_eval"], res["pred_eval"], k_frac=frac), 3)
    rows.append(row)
summary = pd.DataFrame(rows)
summary


### 5. Comparaison au plafond de bruit

Référence `AAV2_viab_noise_ceiling.ipynb` (même CSV) : plafond analytique (Poisson delta-method) +0.744, plafond empirique (split-half + Spearman-Brown, sans hypothèse de forme) +0.819.

In [ ]:
CEILING_ANALYTIC  = 0.744
CEILING_EMPIRICAL = 0.819

rows = []
for tag, res in results.items():
    rows.append({
        "variante": tag, "r held-out": round(res["r"], 3),
        "r / plafond analytique": f"{res['r'] / CEILING_ANALYTIC:.1%}",
        "r / plafond empirique":  f"{res['r'] / CEILING_EMPIRICAL:.1%}",
    })
pd.DataFrame(rows).set_index("variante")


### 6. Le modèle utilise-t-il vraiment `log1p(w)` ?

**6a. Erreur absolue par quantile de confiance.** Si la feature aide, l'écart aux-vs-baseline
devrait être concentré sur les variants les MOINS confiants (là où l'info de profondeur apporte
le plus).

In [ ]:
w_eval = w_all[eval_i]
q_bins = np.quantile(w_eval, np.linspace(0, 1, 11))
q_idx  = np.clip(np.digitize(w_eval, q_bins[1:-1]), 0, 9)

fig, ax = plt.subplots(figsize=(8, 5))
for tag, res in results.items():
    abs_err = np.abs(res["pred_eval"] - res["y_eval"])
    mae_by_bin = [abs_err[q_idx == b].mean() for b in range(10)]
    ax.plot(range(1, 11), mae_by_bin, "o-", label=tag)
ax.set_xlabel("décile de confiance w (1 = le moins confiant, 10 = le plus confiant)")
ax.set_ylabel("erreur absolue moyenne (held-out)")
ax.set_title("AAV2 viab -- erreur par décile de confiance, baseline vs aux")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


**6b. Dépendance partielle.** Séquences fixées (choisies parmi les plus confiantes de l'eval set, réparties sur toute la gamme de `y` réel -- pour que leur vraie valeur soit elle-même fiable), on fait varier artificiellement `log1p(w)` en entrée et on regarde comment la prédiction du modèle *aux* bouge. Hypothèse testée : à confiance basse, la prédiction devrait se rapprocher de la moyenne globale d'entraînement (rétrécissement/shrinkage) ; à confiance haute, elle devrait s'écarter vers la vraie valeur de la séquence.

In [ ]:
model_aux = models["aux (log1p w)"]

# ancres : parmi les 25% les plus confiants de l'eval set, une séquence par decile de y reel
hi_conf = eval_i[w_eval >= np.quantile(w_eval, 0.75)]
y_hi_conf = y_all[hi_conf]
anchor_targets = np.quantile(y_hi_conf, np.linspace(0.05, 0.95, 8))
anchors = [hi_conf[np.argmin(np.abs(y_hi_conf - t))] for t in anchor_targets]

grid = np.linspace(np.percentile(logw_all, 1), np.percentile(logw_all, 99), 30)
y_train_mean = float(y_fit.mean())

fig, ax = plt.subplots(figsize=(8, 5.5))
cmap = plt.cm.viridis(np.linspace(0, 1, len(anchors)))
for c, s in zip(cmap, anchors):
    x_fixed = oh([s])[0]
    X_grid = np.concatenate([np.tile(x_fixed, (len(grid), 1)),
                              grid.reshape(-1, 1).astype(np.float32)], axis=1)
    pred_grid = batched_predict(model_aux, X_grid)
    ax.plot(grid, pred_grid, color=c, lw=1.6)
    ax.axhline(y_all[s], color=c, ls=":", lw=0.8, alpha=0.6)

ax.axhline(y_train_mean, color="k", ls="--", lw=1.2, label="moyenne cible (train)")
ax.set_xlabel("log1p(w) -- confiance simulée (bas = peu confiant, haut = très confiant)")
ax.set_ylabel("log enrichment prédit")
ax.set_title("Dépendance partielle -- 8 séquences fixées, log1p(w) balayé artificiellement\n"
              "(pointillés fins = vraie valeur de chaque séquence)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 7. Notes

- Comparaison strictement appariée (même split fit/eval/train/val, seed=0) -- l'écart de `r` entre
  baseline et aux reflète uniquement l'ajout de `log1p(w)`.
- §6a répond à "où" l'aide se concentre (si elle existe) ; §6b répond à "comment" -- si les courbes
  de dépendance partielle convergent vers la moyenne à gauche (confiance basse) et s'écartent vers
  la vraie valeur à droite (confiance haute), le modèle a effectivement appris un comportement de
  rétrécissement dépendant de la confiance, pas juste capté une corrélation accessoire.
- `w` (donc `log1p(w)`) est calculable pour n'importe quelle séquence de la librairie DÉJÀ
  MESURÉE -- mais pas pour un variant jamais testé en labo. Cette feature n'est donc utile que pour
  re-prédire/débruiter des mesures existantes, pas pour scorer un variant hors librairie (contexte
  à garder en tête si l'usage aval est la recherche du meilleur variant parmi les 20^7 possibles,
  pas seulement le débruitage de la librairie mesurée).